In [ ]:
## Cell 0 — 환경 고정 + 드라이브 (L4·A100 모두 ~2분, 대부분 pip)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers==5.17.0', 'accelerate==1.14.0'])
from google.colab import drive
drive.mount('/content/drive')

import os, re, ast, json, string, unicodedata, hashlib, fnmatch, time
from pathlib import Path
import numpy as np, pandas as pd, torch, transformers, accelerate, sklearn
print(f'torch {torch.__version__} | transformers {transformers.__version__} | '
      f'accelerate {accelerate.__version__} | sklearn {sklearn.__version__} | numpy {np.__version__}')
print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')
assert transformers.__version__ == '5.17.0'

Mounted at /content/drive
torch 2.11.0+cu130 | transformers 5.17.0 | accelerate 1.14.0 | sklearn 1.6.1 | numpy 2.1.3
GPU NVIDIA A100-SXM4-40GB


In [ ]:
## Cell 1 — 입력·채점기·K_strict (CPU, <1분)
DS  = Path('/content/drive/MyDrive/Grad_share/data_share')
E15 = DS/'E15'
OUT = DS/'E18_cplus'; OUT.mkdir(exist_ok=True)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
MAN = json.load(open(E15/'manifest.json'))
W_TAG = MAN['sha_w'][:8]

# (1-1) behavior + fold (질문 순서·C± 짝·fold 정합)
BEH = DS/'P9/hidden_v2/p8s_behavior_v2.csv'
F  = pd.read_csv(BEH).reset_index(drop=True)
FO = pd.read_parquet(E15/'folds.parquet')
assert len(F) == 1224 and (FO.question.to_numpy() == F.question.to_numpy()).all()
F['c'] = F.ctx_label.astype(str).str.strip().map({'C+': 1, 'C-': 0, 'C−': 0})
assert F.c.notna().all()
C, FOLD = F.c.to_numpy().astype(int), FO.fold.to_numpy()
QID = pd.factorize(F.question)[0]
g = F.groupby('question').c
assert F.question.nunique() == 612 and g.size().eq(2).all() and g.sum().eq(1).all()
assert pd.Series(FOLD).groupby(QID).nunique().eq(1).all()          # C± 짝 같은 fold
print('(1-1) sha 일치 | behavior:', sha(BEH) == MAN['sha_behavior'],
      '| folds:', sha(E15/'folds.parquet') == MAN['sha_folds'],
      '| w_v4:', sha(E15/'w_v4.npz')[:8] == W_TAG)
print('      columns:', list(F.columns))

# (1-2) 채점기: trunc_fixed + strict EM (E17 Cell 1과 동일)
PUNCT, ART = set(string.punctuation), {'a', 'an', 'the'}
def norm(s):
    s = unicodedata.normalize('NFKC', str(s)).lower()
    s = ''.join(ch for ch in s if ch not in PUNCT)
    return ' '.join(t for t in s.split() if t not in ART)
_SPLIT = re.compile(r'(?<![A-Z])(?<!\bDr)(?<!\bMr)(?<!\bMs)(?<!\bSt)(?<!\bJr)(?<!\bSr)(?<!\bMrs)\.\s+')
def trunc_fixed(raw):
    s = str(raw).strip().split('\n')[0].strip()
    s = re.sub(r'^(answer|the answer is)\s*:?\s*', '', s, flags=re.I)
    s = _SPLIT.split(s)[0].strip().strip('"\'')
    return re.sub(r'[.,;:\s]+$', '', s).strip()
def parse_gold(x):
    s = str(x)
    for f in (ast.literal_eval, json.loads):
        try:
            v = f(s)
            if isinstance(v, (list, tuple)): return [str(t) for t in v]
        except Exception:
            pass
    return [t.strip() for t in s.split('|') if t.strip()]
GCOL = 'gold_list' if 'gold_list' in F.columns else 'gold'
GL = [parse_gold(x) for x in F[GCOL]]
assert all(len(x) for x in GL)
em_of = lambda p, gl: int(norm(p) in {norm(x) for x in gl})
assert [trunc_fixed(x) for x in ['Dr. Dre', 'T.S. Eliot', 'Ronald D. Moore', 'Paris. It is']] \
       == ['Dr. Dre', 'T.S. Eliot', 'Ronald D. Moore', 'Paris']

# (1-3) K_strict = cb_v4 closed-book 답의 strict EM (N 보고 §13②: 주 K)
CB = pd.read_parquet(E15/'cb_v4.parquet').drop_duplicates('question').set_index('question')
cb_raw = F.question.map(CB.cb_raw); assert cb_raw.notna().all()
K = np.array([em_of(trunc_fixed(r), GL[i]) for i, r in enumerate(cb_raw)])
CELL = np.where(K == 1, np.where(C == 1, 'K+C+', 'K+C−'), np.where(C == 1, 'K−C+', 'K−C−'))
print(f'(1-2) gold 컬럼 {GCOL} | (1-3) K_strict K+ 질문 {K[C == 1].sum()} (N 보고 316)')
print(pd.crosstab(CELL, FOLD))

(1-1) sha 일치 | behavior: False | folds: True | w_v4: True
      columns: ['question', 'gold', 'prop', 'kside', 'knows', 'A_cb', 'ctx_id', 'ctx_title', 's_wiki_title', 'orig_gold', 'ctx_label', 'ctx_text', 'answer_surface', 'ans_start', 'ans_end', 'wrong_answer', 'gold_list', 'kside_k', 'kside_new', 'kside_corr', 'k_agree', 'cell_k', 'c']
(1-2) gold 컬럼 gold_list | (1-3) K_strict K+ 질문 316 (N 보고 316)
col_0   0   1   2   3   4
row_0                    
K+C+   67  62  64  64  59
K+C−   67  62  64  64  59
K−C+   56  61  58  58  63
K−C−   56  61  58  58  63


In [ ]:
## Cell 2 — C+ 보호의 상한 (CPU, <1분, 생성 없음 — E15 생성물 재사용)
ALLG = sorted(p.name[4:-8] for p in E15.glob('gen_*.parquet'))
def find(pat):
    m = [t for t in ALLG if fnmatch.fnmatch(t, pat)]
    assert len(m) == 1, (pat, m)
    return m[0]
def load(tag):
    d = pd.read_parquet(E15/f'gen_{tag}.parquet').set_index('row').reindex(range(len(F)))
    assert d.em.notna().all(), tag
    return d

G = {0.0: load(find('base'))}
for r, pat in [(0.02, 'w_r0.02_xfit_'), (0.04, 'w_r0.04_xfit_'), (0.06, 'A0_r0.06_xfit_'),
               (0.08, 'w_r0.08_xfit_'), (0.10, 'w_r0.1*_xfit_')]:
    G[r] = load(find(pat + W_TAG))
HI = {k: load(find(f'hi_r0.12_w{k}_{W_TAG}')) for k in range(5)}
G[0.12] = pd.concat([HI[FOLD[i]].iloc[[i]] for i in range(len(F))])   # 행마다 자기 fold w

# (2-0) 관문: 채점기 재현 (base raw 재채점 vs 저장 em)
re_em = np.array([em_of(trunc_fixed(r), GL[i]) for i, r in enumerate(G[0.0].raw)])
print('(2-0) 채점기 불일치:', int((re_em != G[0.0].em.to_numpy()).sum()), '(기대 0)')
print('      E15 gen 중 ρ≥0.2 후보:', [t for t in ALLG if re.search(r'r0\.(2|3)', t)])

def boot(delta, mask, n=2000, seed=0):                 # 질문 단위 cluster bootstrap (E15 Cell 9)
    d = pd.DataFrame({'d': delta[mask], 'q': QID[mask]})
    s = d.groupby('q').d.agg(['sum', 'count'])
    S, N_ = s['sum'].to_numpy(), s['count'].to_numpy()
    idx = np.random.default_rng(seed).integers(0, len(S), (n, len(S)))
    est = S[idx].sum(1) / N_[idx].sum(1)
    return round(float(delta[mask].mean()), 4), np.quantile(est, [.025, .975]).round(4).tolist()

EM = {r: d.em.to_numpy().astype(float) for r, d in G.items()}
E0, EA = EM[0.0], EM[0.06]
ALL, CM, CP = np.ones(len(F), bool), C == 0, C == 1

# (2-1) ρ별 C−/C+ 효과와 C+ 손상 행
rows = []
for r in sorted(EM):
    if r == 0: continue
    dmg = CP & (E0 == 1) & (EM[r] == 0); res = CP & (E0 == 0) & (EM[r] == 1)
    rows.append(dict(rho=r, Cm=boot(EM[r] - E0, CM), Cp=boot(EM[r] - E0, CP),
                     dmg=int(dmg.sum()), res=int(res.sum()),
                     dmg_KmCp=int((dmg & (CELL == 'K−C+')).sum())))
print('\n(2-1) vs base  [C− Δ / C+ Δ / C+ 손상·회복 행 / 손상 중 K−C+]')
for x in rows:
    print(f"  ρ{x['rho']:.2f}  C− {x['Cm']}  C+ {x['Cp']}  손상 {x['dmg']} 회복 {x['res']}  K−C+ {x['dmg_KmCp']}")

# (2-2) 상한 vs A0
#  P_C(ρ): C−엔 ρ, C+엔 base  → C를 완벽히 판별하는 늦은 gate의 상한
#  P_D(ρ): C−엔 ρ, C+엔 max(base, ρ) → 손상 행만 완벽히 되돌리는 모니터의 상한 (행별 max라 낙관적)
print('\n(2-2) 상한 − A0  [전체]')
lim = []
for r in [0.08, 0.10, 0.12]:
    PC = np.where(CM, EM[r], E0); PD = np.where(CM, EM[r], np.maximum(E0, EM[r]))
    lc, ld = boot(PC - EA, ALL), boot(PD - EA, ALL)
    lim.append(dict(rho=r, PC=lc[0], PC_ci=lc[1], PD=ld[0], PD_ci=ld[1]))
    print(f'  ρ{r:.2f}  P_C {lc}   P_D {ld}')
pd.DataFrame(lim).to_csv(OUT/'e18_ceiling.csv', index=False)
best = max(x['PC'] for x in lim)
print(f'\n(2-3) P_C 최대 {best*100:+.2f}%p → 기준 +2%p', '통과' if best >= 0.02 else '미달')

(2-0) 채점기 불일치: 0 (기대 0)
      E15 gen 중 ρ≥0.2 후보: []

(2-1) vs base  [C− Δ / C+ Δ / C+ 손상·회복 행 / 손상 중 K−C+]
  ρ0.02  C− (0.0686, [0.0474, 0.0882])  C+ (-0.0016, [-0.0098, 0.0065])  손상 4 회복 3  K−C+ 3
  ρ0.04  C− (0.1013, [0.0768, 0.1258])  C+ (0.0049, [-0.0065, 0.0163])  손상 6 회복 9  K−C+ 5
  ρ0.06  C− (0.1356, [0.1078, 0.1634])  C+ (0.0, [-0.018, 0.0164])  손상 15 회복 15  K−C+ 10
  ρ0.08  C− (0.1569, [0.1275, 0.1895])  C+ (-0.0147, [-0.0359, 0.0065])  손상 27 회복 18  K−C+ 18
  ρ0.10  C− (0.1699, [0.1389, 0.201])  C+ (-0.0376, [-0.0654, -0.0114])  손상 47 회복 24  K−C+ 31
  ρ0.12  C− (0.1699, [0.1356, 0.2042])  C+ (-0.0556, [-0.0866, -0.0277])  손상 61 회복 27  K−C+ 42

(2-2) 상한 − A0  [전체]
  ρ0.08  P_C (0.0106, [-0.0008, 0.0221])   P_D (0.0253, [0.0155, 0.0343])
  ρ0.10  P_C (0.0172, [0.0041, 0.0294])   P_D (0.0368, [0.0245, 0.049])
  ρ0.12  P_C (0.0172, [0.0025, 0.0319])   P_D (0.0392, [0.0253, 0.0539])

(2-3) P_C 최대 +1.72%p → 기준 +2%p 미달


In [ ]:
## Cell 2 (추가) — C− 여지 분해: K+C−를 얼마나 못 고치나
KPCM = CM & (CELL == 'K+C−')
print('\n(2-4) C− 천장 (K+C− 비율 = CB 수준):', round(KPCM.sum() / CM.sum(), 4))
for r in [0.06, 0.12]:
    fail = KPCM & (EM[r] == 0)
    print(f'  ρ{r:.2f}  C− EM {EM[r][CM].mean():.4f} | K+C− 정답률 {EM[r][KPCM].mean():.4f} '
          f'| K+C− 실패 {int(fail.sum())}행 | K−C− 정답 {int((CM & (CELL == "K−C−") & (EM[r] == 1)).sum())}행')


(2-4) C− 천장 (K+C− 비율 = CB 수준): 0.5163
  ρ0.06  C− EM 0.3954 | K+C− 정답률 0.6487 | K+C− 실패 111행 | K−C− 정답 37행
  ρ0.12  C− EM 0.4297 | K+C− 정답률 0.6899 | K+C− 실패 98행 | K−C− 정답 45행


In [ ]:
## Cell 3 — K+C− 저항 행 진단 (CPU, <1분, 생성 없음)
from sklearn.metrics import roc_auc_score
pd.set_option('display.max_colwidth', 40); pd.set_option('display.width', 200)

KP  = CELL == 'K+C−'
FC  = {r: G[r].fc.to_numpy().astype(int) for r in EM}
cbp = np.array([trunc_fixed(x) for x in cb_raw])
P   = {r: np.array([trunc_fixed(x) for x in G[r].raw]) for r in [0.0, 0.06, 0.12]}

# (3-1) K+C− 316행 분류 (base → ρ0.12)
grp = np.full(len(F), '', object)
grp[KP & (E0 == 1) & (EM[0.12] == 1)] = 'keep'      # 원래 맞고 유지
grp[KP & (E0 == 0) & (EM[0.12] == 1)] = 'flip'      # steering으로 회복
grp[KP & (E0 == 1) & (EM[0.12] == 0)] = 'lost'      # steering으로 망침
grp[KP & (E0 == 0) & (EM[0.12] == 0) & (FC[0.12] == 1)] = 'S_ctx'   # 여전히 문맥 답
grp[KP & (E0 == 0) & (EM[0.12] == 0) & (FC[0.12] == 0)] = 'S_oth'   # 문맥 답은 떠났지만 오답
print('(3-1)', pd.Series(grp[KP]).value_counts().to_dict())

# (3-2) 그룹별 ρ 곡선: 문맥 추종(FC)과 EM이 강도에 따라 움직이는가
print('\n(3-2) 그룹별 FC / EM by ρ')
for gname in ['flip', 'S_ctx', 'S_oth']:
    m = grp == gname
    fc = ' '.join(f'{FC[r][m].mean():.2f}' for r in sorted(EM))
    em = ' '.join(f'{EM[r][m].mean():.2f}' for r in sorted(EM))
    print(f'  {gname:5s} n={m.sum():3d} | FC {fc} | EM {em}')
print('  (ρ 순서:', sorted(EM), ')')

# (3-3) 기억 강도 proxy: k_agree (closed-book 샘플 5회 중 정답 비율) — flip vs 저항
RES = np.isin(grp, ['S_ctx', 'S_oth']); FL = grp == 'flip'
ka = F.k_agree.to_numpy().astype(float)
def auc_within_prop(score, pos, neg):
    out, w = [], []
    for p in np.unique(F.prop):
        m = (F.prop.to_numpy() == p) & (pos | neg)
        if (pos & m).sum() >= 3 and (neg & m).sum() >= 3:
            out.append(roc_auc_score(pos[m].astype(int), score[m])); w.append(m.sum())
    return round(float(np.average(out, weights=w)), 3), len(out)
m = FL | RES
print(f'\n(3-3) k_agree 평균 flip {ka[FL].mean():.3f} / S_ctx {ka[grp=="S_ctx"].mean():.3f} / S_oth {ka[grp=="S_oth"].mean():.3f}')
print(f'      AUROC(k_agree → flip vs 저항) 전체 {roc_auc_score(FL[m].astype(int), ka[m]):.3f}'
      f' | prop 고정 {auc_within_prop(ka, FL, RES)}')

# (3-4) prop별 저항률 + 문맥 길이
d = pd.DataFrame({'prop': F.prop, 'g': grp, 'len': F.ctx_text.str.len()})[KP & (E0 == 0)]
t = pd.crosstab(d.prop, d.g); t['resist_rate'] = ((t.get('S_ctx', 0) + t.get('S_oth', 0)) / t.sum(1)).round(2)
print('\n(3-4) base 오답 K+C− 행의 prop별 결과\n', t)
print('      문맥 길이 중앙값 flip', int(d[d.g == 'flip'].len.median()),
      '| S_ctx', int(d[d.g == 'S_ctx'].len.median()), '| S_oth', int(d[d.g == 'S_oth'].len.median()))

# (3-5) 예시: S_oth는 무엇을 내는가 (CB답과 같은가 / 형식 문제인가)
so = np.where(grp == 'S_oth')[0]
same_cb = np.array([norm(P[0.12][i]) == norm(cbp[i]) for i in so])
print(f'\n(3-5) S_oth {len(so)}행 중 ρ0.12 답 == CB답(정규화): {same_cb.sum()}  (K+라 CB답은 EM 정답 → 0이어야 정상)')
ex = pd.DataFrame({'q': F.question.to_numpy()[so], 'gold': [GL[i][0] for i in so],
                   'ctx': F.answer_surface.to_numpy()[so], 'CB': cbp[so],
                   'base': P[0.0][so], 'ρ.12': P[0.12][so]})
print(ex.head(12).to_string())
sc = np.where(grp == 'S_ctx')[0]
print('\n(3-5b) S_ctx 예시'); print(pd.DataFrame({'q': F.question.to_numpy()[sc], 'gold': [GL[i][0] for i in sc],
      'ctx': F.answer_surface.to_numpy()[sc], 'CB': cbp[sc], 'ρ.12': P[0.12][sc]}).head(8).to_string())

pd.DataFrame({'row': np.arange(len(F)), 'grp': grp}).to_parquet(OUT/'e18_kpcm_groups.parquet', index=False)

(3-1) {'keep': 134, 'flip': 84, 'S_oth': 68, 'S_ctx': 23, 'lost': 7}

(3-2) 그룹별 FC / EM by ρ
  flip  n= 84 | FC 0.75 0.50 0.36 0.21 0.13 0.02 0.00 | EM 0.00 0.38 0.57 0.74 0.82 0.94 1.00
  S_ctx n= 23 | FC 1.00 1.00 1.00 1.00 1.00 1.00 1.00 | EM 0.00 0.00 0.00 0.00 0.00 0.00 0.00
  S_oth n= 68 | FC 0.60 0.49 0.47 0.31 0.18 0.12 0.00 | EM 0.00 0.03 0.03 0.06 0.07 0.04 0.00
  (ρ 순서: [0.0, 0.02, 0.04, 0.06, 0.08, 0.1, 0.12] )

(3-3) k_agree 평균 flip 0.905 / S_ctx 0.887 / S_oth 0.809
      AUROC(k_agree → flip vs 저항) 전체 0.595 | prop 고정 (0.55, 7)

(3-4) base 오답 K+C− 행의 prop별 결과
 g               S_ctx  S_oth  flip  resist_rate
prop                                           
author              2      8    14         0.42
capital             0      8    18         0.31
capital of          0      2     3         0.40
color               2      1     0         1.00
composer            5      2     1         0.88
country             0      1    10         0.09
director            2      8     7  

In [ ]:
## Cell 4 — S_oth 자동 분류 + 측정 보정 민감도 (CPU, 수 초)
STOP = {'of', 'and', 'in', 'de', 'la', 'le'}
toks    = lambda s: set(norm(s).split()) - STOP
nospace = lambda s: norm(s).replace(' ', '')
ABST = re.compile(r'\b(unknown|not (mentioned|stated|provided|given)|no (information|answer)|cannot|none|n/a)\b', re.I)
NTOK = {r: G[r].n_tok.to_numpy() for r in EM}

def cat(i, r):
    p, ctxa = P[r][i], str(F.answer_surface.iloc[i])
    tp, tc = toks(p), toks(ctxa)
    tg = set().union(*[toks(x) for x in GL[i] + [cbp[i]]])
    if NTOK[r][i] > 12 or len(norm(p).split()) > 8: return 'degen'
    if norm(p) == '' or ABST.search(p):             return 'abstain'
    if nospace(p) == nospace(ctxa) or (tp and tp <= tc): return 'ctx_partial'   # FC 미탐
    if tp and tp <= tg:                             return 'gold_partial'      # 정답 일부·상위어
    if tp & tg and tp & tc:                         return 'blend'
    if tp & tg:                                     return 'gold_overlap'
    return 'other'

ORDER = ['ctx_partial', 'gold_partial', 'blend', 'gold_overlap', 'other', 'abstain', 'degen']
so = np.where(grp == 'S_oth')[0]
for r in [0.0, 0.06, 0.12]:
    vc = pd.Series([cat(i, r) for i in so]).value_counts().reindex(ORDER, fill_value=0)
    print(f'(4-1) S_oth {len(so)}행 @ρ{r:.2f}:', vc.to_dict())

# (4-2) 유형별 예시 (ρ0.12)
c12 = np.array([cat(i, 0.12) for i in so])
for k in ORDER:
    ii = so[c12 == k][:3]
    if len(ii):
        print(f'\n  [{k}]', ' | '.join(f'{P[0.12][i]!r} (gold {GL[i][0]!r}, ctx {F.answer_surface.iloc[i]!r})' for i in ii))

# (4-3) 측정 보정 민감도: ctx_partial → S_ctx, gold_partial → 정답으로 볼 때 K+C− 실패 수
n_ctx  = (grp == 'S_ctx').sum() + (c12 == 'ctx_partial').sum()
n_gp   = (c12 == 'gold_partial').sum()
n_true = len(so) - (c12 == 'ctx_partial').sum() - n_gp
print(f'\n(4-3) ρ0.12 K+C− 실패 98행 재분해 → 진짜 미이동(S_ctx+ctx_partial) {n_ctx} | '
      f'채점 문제(gold_partial) {n_gp} | 이동했으나 오답(blend·overlap·other·abstain·degen) {n_true}')
pd.DataFrame({'row': so, 'cat_r012': c12}).to_parquet(OUT/'e18_soth_cat.parquet', index=False)

(4-1) S_oth 68행 @ρ0.00: {'ctx_partial': 42, 'gold_partial': 5, 'blend': 0, 'gold_overlap': 2, 'other': 19, 'abstain': 0, 'degen': 0}
(4-1) S_oth 68행 @ρ0.06: {'ctx_partial': 25, 'gold_partial': 10, 'blend': 1, 'gold_overlap': 2, 'other': 27, 'abstain': 2, 'degen': 1}
(4-1) S_oth 68행 @ρ0.12: {'ctx_partial': 6, 'gold_partial': 8, 'blend': 1, 'gold_overlap': 2, 'other': 46, 'abstain': 4, 'degen': 1}

  [ctx_partial] np.str_('T.S. Eliot') (gold 'Thomas Hardy', ctx 'T.\xa0S. Eliot') | np.str_('Anabella') (gold 'Gruoch of Scotland', ctx 'Anabella Drummond') | np.str_('Speedway') (gold 'association football', ctx 'motorcycle speedway')

  [gold_partial] np.str_('Milton') (gold 'Milton Friedman', ctx 'Joel Spolsky') | np.str_('Electronic') (gold 'electronic dance music', ctx 'industrial music') | np.str_('Industrial') (gold 'industrial metal', ctx 'science')

  [blend] np.str_("People's Republic of Algeria") (gold 'Algeria', ctx "People's Republic of China")

  [gold_overlap] np.str_('Nancy Hur

In [ ]:
## Cell 5 — 모델 로드 + w_v4 훅 + golden 확인 (A100 ~3분 / L4 ~5분, 대부분 다운로드)
from getpass import getpass
from huggingface_hub import login, model_info
from transformers import AutoTokenizer, AutoModelForCausalLM
login(token=getpass('HF token: ').strip(), add_to_git_credential=False)
MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
REV = model_info(MODEL_ID).sha; assert REV.startswith('0e9e39f'), REV
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REV)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REV, dtype=torch.bfloat16,
                                             attn_implementation='sdpa', device_map='cuda').eval()
LAYERS = model.model.layers

SYSTEM_PROMPT = ("You are a question answering system. "
    "Answer the question based on the given context. "
    "Output the answer string itself and nothing else: no sentence, no explanation, "
    "no punctuation, no quotes. Keep it as short as possible (a name or a term).")
USER_TMPL = 'Context: {ctx}\nQuestion: {q}\nAnswer:'
def prompt_ids(i):
    m = [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user', 'content': USER_TMPL.format(ctx=F.ctx_text.iloc[i], q=F.question.iloc[i])}]
    return tok.apply_chat_template(m, add_generation_prompt=True, tokenize=True,
                                   return_tensors='pt', return_dict=True)['input_ids'][0]

# (5-1) w_v4 훅: L14–18, 전 토큰(prefill+decode), Δ는 fp32로 만들고 bf16으로 더함 (E17-5 재현 조건)
WZ = np.load(E15/'w_v4.npz'); W_, MED_ = WZ['W'].astype(np.float32), WZ['MED'].astype(np.float32)
assert W_.shape == (5, 5, 4096) and MED_.shape == (5, 5)
STEER_L = [14, 15, 16, 17, 18]
_cur = {'d': None}
def _hook(j):
    def h(mod, inp, out):
        d = _cur['d']
        if d is None: return out
        return (out[0] + d[j],) + tuple(out[1:]) if isinstance(out, tuple) else out + d[j]
    return h
HOOKS = [LAYERS[l].register_forward_hook(_hook(j)) for j, l in enumerate(STEER_L)]
_DC = {}
def set_rho(i, rho):
    if rho == 0: _cur['d'] = None; return
    key = (int(FOLD[i]), rho)
    if key not in _DC:
        _DC[key] = torch.tensor(rho * MED_[key[0]][:, None] * W_[key[0]], dtype=torch.bfloat16, device='cuda')
    _cur['d'] = _DC[key]

# (5-2) golden: K+C− 8행, bs1 greedy → E15 저장 출력과 예측 문자열 비교 (bs16 저장본이라 1–2행 차이는 허용)
@torch.inference_mode()
def gen1(i, rho, n=24):
    set_rho(i, rho); ids = prompt_ids(i).unsqueeze(0).cuda()
    out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=n, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    _cur['d'] = None
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True, clean_up_tokenization_spaces=False)
gi = np.where(KP)[0][::40][:8]
for r in [0.0, 0.06]:
    ok = sum(trunc_fixed(gen1(i, r)) == P[r][i] for i in gi)
    print(f'(5-2) golden ρ{r:.2f}: {ok}/8 일치')

HF token: ··········


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

(5-2) golden ρ0.00: 8/8 일치
(5-2) golden ρ0.06: 8/8 일치


In [ ]:
## Cell 6 — teacher forcing: CB답·문맥답 log확률 (K+C− 316행 × ρ 3개 × 답 2개, A100 ~2분 / L4 ~5분)
@torch.inference_mode()
def tf_score(i, target, rho):
    set_rho(i, rho)
    p = prompt_ids(i); t = tok(str(target), add_special_tokens=False, return_tensors='pt')['input_ids'][0]
    ids = torch.cat([p, t]).unsqueeze(0).cuda()
    lg = model(ids, logits_to_keep=len(t) + 1).logits[0, :-1].float()
    lp = torch.log_softmax(lg, -1); tt = t.cuda()
    tl = lp[torch.arange(len(t)), tt]
    rank1 = int((lp[0] > lp[0, tt[0]]).sum()) + 1
    _cur['d'] = None
    return tl.mean().item(), rank1

kp_rows = np.where(KP)[0]
recs = []
t0 = time.time()
for i in kp_rows:
    for r in [0.0, 0.06, 0.12]:
        cb_lp, cb_rk = tf_score(i, cbp[i], r)
        cx_lp, cx_rk = tf_score(i, F.answer_surface.iloc[i], r)
        recs.append(dict(row=i, rho=r, cb_lp=cb_lp, cb_rank=cb_rk, cx_lp=cx_lp, cx_rank=cx_rk))
TF = pd.DataFrame(recs); TF.to_parquet(OUT/'e18_tf_kpcm.parquet', index=False)
print(f'TF 완료 {time.time() - t0:.0f}s')

# (6-1) 그룹 정의: Cell 3·4 결과 사용
cat_map = dict(zip(so, c12))
def g2(i):
    if grp[i] == 'S_ctx' or cat_map.get(i) == 'ctx_partial': return 'stuck'
    if cat_map.get(i) == 'gold_partial': return 'gold_partial'
    if grp[i] == 'S_oth': return 'moved_wrong'
    return grp[i]
TF['g'] = TF.row.map(g2)
w = TF.pivot_table(index=['row', 'g'], columns='rho', values=['cb_lp', 'cx_lp', 'cb_rank']).reset_index()
out = []
for gname in ['flip', 'moved_wrong', 'stuck', 'keep']:
    m = w.g == gname
    dcb = w.loc[m, ('cb_lp', 0.12)] - w.loc[m, ('cb_lp', 0.0)]
    dcx = w.loc[m, ('cx_lp', 0.12)] - w.loc[m, ('cx_lp', 0.0)]
    out.append(dict(g=gname, n=int(m.sum()),
                    dCB_med=round(dcb.median(), 2), dCB_pos=round((dcb > 0).mean(), 2),
                    dCTX_med=round(dcx.median(), 2),
                    CBrank_base=int(w.loc[m, ('cb_rank', 0.0)].median()),
                    CBrank_012=int(w.loc[m, ('cb_rank', 0.12)].median()),
                    CBrank012_le3=round((w.loc[m, ('cb_rank', 0.12)] <= 3).mean(), 2)))
print('\n(6-1) ρ0.12 − base, log확률 평균(nat/토큰)'); print(pd.DataFrame(out).to_string(index=False))

TF 완료 71s

(6-1) ρ0.12 − base, log확률 평균(nat/토큰)
          g   n  dCB_med  dCB_pos  dCTX_med  CBrank_base  CBrank_012  CBrank012_le3
       flip  84     1.56     0.98     -1.82            3           1           1.00
moved_wrong  54     1.87     0.85     -1.36            7           4           0.44
      stuck  29     3.74     1.00     -0.49           14           3           0.55
       keep 134    -0.06     0.25     -1.63            1           1           0.99


In [ ]:
## Cell 7 — base 첫 답 토큰 확신도 + 행별 강도 정책 nested 시뮬레이션
##          (prefill 1224회: A100 ~1분 / L4 ~3분, 이후 CPU ~1분)
@torch.inference_mode()
def first_tok_stats(i):
    _cur['d'] = None
    ids = prompt_ids(i).unsqueeze(0).cuda()
    lg = model(ids, logits_to_keep=1).logits[0, -1].float()
    p = torch.softmax(lg, -1); top = torch.topk(p, 2).values
    ent = -(p * torch.log(p.clamp_min(1e-12))).sum().item()
    return top[0].item(), (torch.log(top[0]) - torch.log(top[1])).item(), ent
t0 = time.time()
S = np.array([first_tok_stats(i) for i in range(len(F))])
SIG = {'maxA': S[:, 0], 'margA': S[:, 1], 'entA': S[:, 2]}
pd.DataFrame(SIG).assign(row=np.arange(len(F))).to_parquet(OUT/'e18_firsttok.parquet', index=False)
print(f'(7-0) {time.time() - t0:.0f}s')

# (7-1) 신호가 "필요 강도"를 읽는가: K+C− base 오답 중 flip vs 저항, 그리고 C+ 손상(ρ0.12) vs 비손상
DMG = CP & (E0 == 1) & (EM[0.12] == 0); NDMG = CP & ~DMG
print('\n(7-1) AUROC [flip vs 저항(K+C−) | 전체, prop 고정]  /  [C+ 손상 vs 비손상]')
for k, s in SIG.items():
    m1, m2 = FL | RES, CP
    a1 = roc_auc_score(RES[m1].astype(int), s[m1]); a2 = roc_auc_score(DMG[m2].astype(int), s[m2])
    print(f'  {k:5s}  저항 {a1:.3f} | prop고정 {auc_within_prop(s, RES, FL)}   C+손상 {a2:.3f} | prop고정 {auc_within_prop(s, DMG, NDMG)}')
    print(f'         중앙값  flip {np.median(s[FL]):.2f} / 저항 {np.median(s[RES]):.2f} / C+손상 {np.median(s[DMG]):.2f} / C+비손상 {np.median(s[NDMG]):.2f}')

# (7-2) nested 정책: 상수 ρ vs 신호 2단 ρ (선택 규칙 = train C+ Δ ≥ −0.5%p 중 train EM 최대)
RG = [0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12]
EMA, ar = np.stack([EM[r] for r in RG]), np.arange(len(F))
def pol(sig, tau, lo, hi, sign):
    return EMA[np.where(sign * sig > sign * tau, hi, lo), ar]
CONST = [(k, k, 1, 0.5) for k in range(1, 7)]
TWO = [(lo, hi, sg, q) for lo in range(1, 7) for hi in range(lo + 1, 7)
       for sg in (1, -1) for q in (0.2, 0.35, 0.5, 0.65, 0.8)]
def nested(cset, sig):
    out, pick = np.zeros(len(F)), []
    for f in range(5):
        tr, te = FOLD != f, FOLD == f
        best = None
        for lo, hi, sg, q in cset:
            tau = np.quantile(sig[tr], q); e = pol(sig, tau, lo, hi, sg)
            if (e - E0)[tr & CP].mean() < -0.005: continue
            if best is None or e[tr].mean() > best[0]: best = (e[tr].mean(), lo, hi, sg, q, tau)
        _, lo, hi, sg, q, tau = best
        out[te] = pol(sig, tau, lo, hi, sg)[te]; pick.append(f'{RG[lo]}/{RG[hi]}/{"+" if sg > 0 else "−"}/{q}')
    return out, pick

base_c, pk_c = nested(CONST, SIG['maxA'])
print(f'\n(7-2) 상수 ρ nested  EM {base_c.mean():.4f}  선택 {pk_c}')
res7 = []
for k, s in SIG.items():
    e, pk = nested(TWO, s)
    d, cp = boot(e - base_c, ALL), boot(e - E0, CP)
    res7.append(dict(sig=k, EM=round(e.mean(), 4), vs_const=d, Cp_vs_base=cp, pick=pk))
    print(f'  {k:5s}  EM {e.mean():.4f} | − 상수 {d} | C+ vs base {cp}\n         선택 {pk}')
print(f'  참고: row-oracle(행별 최선 ρ) EM {EMA.max(0).mean():.4f}')
pd.DataFrame(res7).to_csv(OUT/'e18_dose_policy.csv', index=False)

(7-0) 46s

(7-1) AUROC [flip vs 저항(K+C−) | 전체, prop 고정]  /  [C+ 손상 vs 비손상]
  maxA   저항 0.583 | prop고정 (0.665, 7)   C+손상 0.455 | prop고정 (0.377, 9)
         중앙값  flip 0.88 / 저항 0.97 / C+손상 1.00 / C+비손상 1.00
  margA  저항 0.587 | prop고정 (0.658, 7)   C+손상 0.467 | prop고정 (0.398, 9)
         중앙값  flip 2.38 / 저항 3.38 / C+손상 7.12 / C+비손상 7.88
  entA   저항 0.426 | prop고정 (0.357, 7)   C+손상 0.559 | prop고정 (0.63, 9)
         중앙값  flip 0.49 / 저항 0.17 / C+손상 0.01 / C+비손상 0.01

(7-2) 상수 ρ nested  EM 0.5915  선택 ['0.06/0.06/+/0.5', '0.06/0.06/+/0.5', '0.06/0.06/+/0.5', '0.06/0.06/+/0.5', '0.04/0.04/+/0.5']
  maxA   EM 0.5931 | − 상수 (0.0016, [-0.0057, 0.009]) | C+ vs base (-0.0131, [-0.0327, 0.0065])
         선택 ['0.02/0.08/−/0.8', '0.06/0.12/−/0.5', '0.06/0.1/+/0.8', '0.02/0.08/−/0.65', '0.02/0.06/−/0.65']
  margA  EM 0.5931 | − 상수 (0.0016, [-0.0049, 0.0082]) | C+ vs base (-0.0016, [-0.0196, 0.0163])
         선택 ['0.06/0.12/−/0.2', '0.06/0.12/−/0.2', '0.06/0.1/−/0.2', '0.02/0.08/−/0.65', '0.04/0.1/−/0.35'

**질문: C+를 보호하면 강도를 더 올릴 수 있는가?** → 결과 세 단계

1. C+를 완벽히 보호해도 상한은 +1.7%p. C−가 ρ0.10에서 포화되기 때문 (Cell 2). 제약 요인은 C+가 아니라 C− 포화.
2. 포화 기제 = 같은 강도, 서로 다른 출발점. steering은 실패한 행에서도 기억 확률을 올림 (+1.9~3.7 nat/토큰). 단 CB 답이 3위에서 출발한 행은 1위로 올라가고, 7위·14위에서 출발한 행은 못 올라감 (Cell 6). "빼기만 한다" 가설 기각.
3. 행별 강도에는 큰 여지가 있으나 (row-oracle +7%p), 1-pass에서 관측 가능한 확신도로는 예측되지 않음. 확신도는 필요한 강도보다 C 정보를 먼저 담고 있음 (Cell 7).

**종합**: 세 결과 모두 E17의 결론으로 수렴. 1-pass에 없는 정보는 모델 자신의 closed-book 답.
- 참고: row-oracle EM 0.662 ≈ E17 packed 온라인 0.659. 배치·기제가 달라 paired 비교는 아니며 대략적 비교임.
- packed가 "행별 최적 강도 선택"과 유사한 역할을 CB 분기로 수행한다는 정황으로 사용 가능.

**요약**: 1-pass steering의 강도 상한은 C+ 손상이 아니라 행별 필요 강도의 이질성에서 오며, 이를 1-pass 관측 신호로 예측하지 못했다(사전 기준 미달). 이 이질성을 해소하는 정보가 closed-book 답이다.

### 두 번째 시도 _ slot 예측

In [ ]:
## Cell 8 — 문맥 slot 예측: 1-pass 안의 기억 후보 + 상한 (1224행 prefill, A100 ~1분 / L4 ~3분)
def prompt_str(i):
    m = [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user', 'content': USER_TMPL.format(ctx=F.ctx_text.iloc[i], q=F.question.iloc[i])}]
    return tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False)

# (8-1) 답 span 메타 검증
ok_span = np.array([str(F.ctx_text.iloc[i])[int(F.ans_start.iloc[i]):int(F.ans_end.iloc[i])] == str(F.answer_surface.iloc[i])
                    if pd.notna(F.ans_start.iloc[i]) else False for i in range(len(F))])
print('(8-1) ctx_text[ans_start:ans_end] == answer_surface:', round(ok_span.mean(), 4))

@torch.inference_mode()
def slot_stats(i):
    _cur['d'] = None
    s = prompt_str(i); c0 = s.find(str(F.ctx_text.iloc[i])); assert c0 >= 0
    enc = tok(s, add_special_tokens=False, return_offsets_mapping=True)
    ids = torch.tensor(enc['input_ids']).unsqueeze(0).cuda()
    a0 = c0 + int(F.ans_start.iloc[i])
    j = next(k for k, (st, en) in enumerate(enc['offset_mapping']) if en > a0)   # 답 span 첫 토큰
    lp = torch.log_softmax(model(ids).logits[0, [j - 1, ids.shape[1] - 1]].float(), -1)
    t_ctx = ids[0, j].item()
    return dict(row=i, j=j, n=ids.shape[1], ids_ok=torch.equal(ids[0].cpu(), prompt_ids(i)),
                slot_lp_ctx=lp[0, t_ctx].item(), slot_max=lp[0].max().item(),
                slot_top1=tok.decode([lp[0].argmax().item()]), ctx_tok=tok.decode([t_ctx]))

t0 = time.time()
SL = pd.DataFrame([slot_stats(i) for i in np.where(ok_span)[0]]).set_index('row')
SL.to_parquet(OUT/'e18_slot.parquet'); print(f'(8-0) {time.time() - t0:.0f}s | 토큰화 일치 {SL.ids_ok.mean():.4f}')

# (8-2) slot top1이 기억 후보인가 (CB 답 / 문맥 답 / 그 외)
w0 = lambda s: norm(s).split()[0] if norm(s).split() else ''
def kind(i):
    t = norm(SL.slot_top1[i])
    if not t: return 'none'
    if w0(cbp[i]).startswith(t) or t.startswith(w0(cbp[i])): return 'CB'
    if w0(F.answer_surface.iloc[i]).startswith(t): return 'ctx'
    return 'other'
SL['kind'] = [kind(i) for i in SL.index]
cell_s = pd.Series(CELL, index=range(len(F)))[SL.index]
print('\n(8-2) slot top1 종류 (셀별 비율)'); print(pd.crosstab(cell_s, SL.kind, normalize='index').round(3))

# (8-3) slot surprisal의 판별력: C (C− vs C+), 그리고 C− 안에서 K (기억이 있는가)
r_ = SL.index.to_numpy(); sp = -SL.slot_lp_ctx.to_numpy()          # 높을수록 문맥 답이 의외
FULLm = lambda v: np.full(len(F), np.nan) if v is None else v
spF = np.full(len(F), np.nan); spF[r_] = sp
has = ~np.isnan(spF)
print(f'\n(8-3) AUROC surprisal → C−: 전체 {roc_auc_score(CM[has], spF[has]):.3f} | prop고정 {auc_within_prop(np.nan_to_num(spF), CM & has, CP & has)}')
km = has & CM
print(f'      C− 안에서 → K+: {roc_auc_score((K[km] == 1).astype(int), spF[km]):.3f} | 참고 L13 C probe 0.904 (E15)')

# (8-4) 거친 상한: C−이고 slot이 CB 후보면 CB EM, 아니면 base 유지 / C+는 base (oracle C + oracle span)
ub = E0.copy()
for i in r_:
    if CM[i] and SL.kind[i] == 'CB': ub[i] = K[i]
print(f'\n(8-4) 상한 EM {ub.mean():.4f} | − A0 {boot(ub - EA, ALL)} | 참고 C-oracle(CB 후보) 0.659, packed 온라인 0.659')
print(f'      C− 중 slot=CB 후보 비율 {(SL.kind[SL.index[CM[r_]]] == "CB").mean():.3f}')

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


(8-1) ctx_text[ans_start:ans_end] == answer_surface: 0.8791
(8-0) 42s | 토큰화 일치 1.0000

(8-2) slot top1 종류 (셀별 비율)
kind      CB    ctx   none  other
row_0                            
K+C+   0.566  0.004  0.187  0.243
K+C−   0.500  0.003  0.196  0.301
K−C+   0.083  0.349  0.144  0.424
K−C−   0.091  0.003  0.145  0.760

(8-3) AUROC surprisal → C−: 전체 0.969 | prop고정 (0.967, 16)
      C− 안에서 → K+: 0.625 | 참고 L13 C probe 0.904 (E15)

(8-4) 상한 EM 0.6021 | − A0 (0.0033, [-0.0147, 0.0212]) | 참고 C-oracle(CB 후보) 0.659, packed 온라인 0.659
      C− 중 slot=CB 후보 비율 0.302


In [ ]:
## Cell 8b — 상한 재정의(A0 채움) + 현실 정책 nested (CPU, 수 초)
r_ = SL.index.to_numpy()
KIND = np.full(len(F), 'nospan', object); KIND[r_] = SL.kind.to_numpy()
SMAX = np.full(len(F), -np.inf); SMAX[r_] = SL.slot_max.to_numpy()
SP   = np.full(len(F), -np.inf); SP[r_]   = -SL.slot_lp_ctx.to_numpy()   # 문맥 답 surprisal
HAS  = KIND != 'nospan'
CAN  = np.isin(KIND, ['CB', 'ctx', 'other'])                            # 교체 가능한 토큰(none 제외)
EMSW = np.where(KIND == 'CB', K, np.where(KIND == 'ctx', C, 0)).astype(float)

# (8b-1) oracle C 상한 (나머지는 A0)
ub_cb  = np.where(CM & (KIND == 'CB'), K, EA)        # C−이고 slot=CB인 행만 교체 (종류까지 oracle)
ub_any = np.where(CM & CAN, EMSW, EA)                # C−면 종류 모르고 교체 (C만 oracle)
for nm, u in [('C+종류 oracle', ub_cb), ('C만 oracle', ub_any)]:
    print(f'(8b-1) {nm:12s} EM {u.mean():.4f} | − A0 {boot(u - EA, ALL)} | C− − A0 {boot(u - EA, CM)}')

# (8b-2) 현실 정책: surprisal·slot 확신도로 교체 여부 결정, τ·κ nested (train C+ Δ ≥ −0.5%p 중 EM 최대)
QS  = [0.3, 0.4, 0.5, 0.6, 0.7]
KAP = [-np.inf, np.log(0.1), np.log(0.3), np.log(0.5), np.log(0.7)]
def pol8(tau, kap):
    sw = CAN & (SP > tau) & (SMAX > kap)
    return np.where(sw, EMSW, EA), sw
out8, sw8, pk8 = np.zeros(len(F)), np.zeros(len(F), bool), []
for f in range(5):
    tr, te = FOLD != f, FOLD == f
    best = None
    for q in QS:
        tau = np.quantile(SP[tr & HAS], q)
        for kap in KAP:
            e, sw = pol8(tau, kap)
            if (e - E0)[tr & CP].mean() < -0.005: continue
            if best is None or e[tr].mean() > best[0]: best = (e[tr].mean(), q, kap, tau)
    _, q, kap, tau = best
    e, sw = pol8(tau, kap); out8[te], sw8[te] = e[te], sw[te]
    pk8.append(f'q{q}/p>{np.exp(kap):.2f}')
print(f'\n(8b-2) 현실 정책 nested EM {out8.mean():.4f} | − A0 {boot(out8 - EA, ALL)} | C+ vs base {boot(out8 - E0, CP)}')
print(f'       교체율 C− {sw8[CM].mean():.3f} / C+ {sw8[CP].mean():.3f} | 선택 {pk8}')
print('       셀별 Δ vs A0:', {c: round(float((out8 - EA)[CELL == c].mean()), 4) for c in ['K+C+', 'K+C−', 'K−C+', 'K−C−']})
pd.DataFrame({'row': np.arange(len(F)), 'em_pol8': out8, 'switch': sw8}).to_parquet(OUT/'e18_pol8.parquet', index=False)

(8b-1) C+종류 oracle  EM 0.6422 | − A0 (0.0433, [0.0327, 0.0547]) | C− − A0 (0.0866, [0.0654, 0.1095])
(8b-1) C만 oracle    EM 0.5711 | − A0 (-0.0278, [-0.0449, -0.0098]) | C− − A0 (-0.0556, [-0.0899, -0.0196])


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
## Cell 8b-2 (수정) — 제약을 A0 대비로, 후보 없으면 교체 안 함 + κ의 판별력 진단 (CPU, 수 초)
# (8b-0) slot 확신도가 "기억 답 후보(CB)"와 "엉뚱한 후보(other)"를 가르는가 (C− 행)
mm = CM & np.isin(KIND, ['CB', 'other'])
print(f'(8b-0) C−에서 SMAX → CB vs other AUROC {roc_auc_score((KIND[mm] == "CB").astype(int), SMAX[mm]):.3f}'
      f' | prop고정 {auc_within_prop(SMAX, mm & (KIND == "CB"), mm & (KIND == "other"))}')
for p in [0.1, 0.3, 0.5, 0.7]:
    s = mm & (SMAX > np.log(p))
    print(f'       slot p>{p}: C− 후보 {s.sum():3d}행 중 CB 비율 {(KIND[s] == "CB").mean():.3f}')

out8, sw8, pk8 = np.zeros(len(F)), np.zeros(len(F), bool), []
for f in range(5):
    tr, te = FOLD != f, FOLD == f
    best = (EA[tr].mean(), None, None, np.inf)                 # 기본값: 교체 안 함 (= A0)
    for q in QS:
        tau = np.quantile(SP[tr & HAS], q)
        for kap in KAP:
            e, sw = pol8(tau, kap)
            if (e - EA)[tr & CP].mean() < -0.005: continue      # C+ 손실은 A0 대비 0.5%p까지
            if e[tr].mean() > best[0]: best = (e[tr].mean(), q, kap, tau)
    _, q, kap, tau = best
    e, sw = pol8(tau, kap if kap is not None else np.inf)
    out8[te], sw8[te] = e[te], sw[te]
    pk8.append('none' if q is None else f'q{q}/p>{np.exp(kap):.2f}')
print(f'\n(8b-2) 현실 정책 nested EM {out8.mean():.4f} | − A0 {boot(out8 - EA, ALL)} | C+ vs base {boot(out8 - E0, CP)}')
print(f'       교체율 C− {sw8[CM].mean():.3f} / C+ {sw8[CP].mean():.3f} | 선택 {pk8}')
print('       셀별 Δ vs A0:', {c: round(float((out8 - EA)[CELL == c].mean()), 4) for c in ['K+C+', 'K+C−', 'K−C+', 'K−C−']})
pd.DataFrame({'row': np.arange(len(F)), 'em_pol8': out8, 'switch': sw8}).to_parquet(OUT/'e18_pol8.parquet', index=False)

(8b-0) C−에서 SMAX → CB vs other AUROC 0.791 | prop고정 (0.792, 14)
       slot p>0.1: C− 후보 487행 중 CB 비율 0.376
       slot p>0.3: C− 후보 381행 중 CB 비율 0.451
       slot p>0.5: C− 후보 277행 중 CB 비율 0.552
       slot p>0.7: C− 후보 200행 중 CB 비율 0.655

(8b-2) 현실 정책 nested EM 0.6144 | − A0 (0.0155, [0.0025, 0.0286]) | C+ vs base (-0.0033, [-0.0212, 0.0147])
       교체율 C− 0.328 / C+ 0.005 | 선택 ['q0.4/p>0.70', 'q0.5/p>0.70', 'q0.4/p>0.70', 'q0.5/p>0.70', 'q0.4/p>0.70']
       셀별 Δ vs A0: {'K+C+': -0.0032, 'K+C−': 0.0981, 'K−C+': -0.0034, 'K−C−': -0.0338}


원래 입력 한 번의 forward 안에서 ① C 판별(slot surprisal, AUROC 0.97, 라벨 없음)과 ② 기억 후보(K+C−의 50%)는 얻을 수 있다. 그러나 ③ 그 후보를 믿을지는 1-pass 신호로 0.79까지만 판단된다. ③에 필요한 비교 정보(B_of_A, 선택 AUROC 0.93–0.95)가 closed-book 계산이고, packed가 그것을 1.09배 비용으로 얻는다.

In [ ]:
## Cell 9 — "같은 도착점" 정책 오프라인 검증: 사영·각도로 행별 ρ 결정 (CPU ~1분, 모델 불필요)
##          선행: Cell 0–3 (Cell 7의 SIG는 불필요). 끊긴 런타임이면 0→1→2→3→9
from sklearn.metrics import roc_auc_score
CAP = np.load(E15/'cap_bs1.npz')
print('(9-0) cap_bs1 키:', {k: CAP[k].shape for k in CAP.files})
HW = CAP['HW'].astype(np.float32); assert HW.shape == (len(F), 5, 4096)       # L14–18, 마지막 토큰, 비개입
WZ = np.load(E15/'w_v4.npz'); W_, MED_ = WZ['W'].astype(np.float32), WZ['MED'].astype(np.float32)
assert np.allclose(np.linalg.norm(W_, axis=-1), 1, atol=1e-3)
PROJ = (HW * W_[FOLD]).sum(-1)                    # (n,5) 자기 fold w에 대한 사영 (retain 쪽 위치)
COS  = PROJ / np.linalg.norm(HW, axis=-1)          # (n,5) 각도의 cos
MEDR = MED_[FOLD]                                  # (n,5) Δ = ρ·MED·w → 사영이 ρ·MED만큼 이동

# (9-1) 기하 신호가 "필요 강도"를 읽는가 — Cell 7 확신도와 같은 표
FL  = grp == 'flip'; RES = np.isin(grp, ['S_ctx', 'S_oth'])
DMG = CP & (E0 == 1) & (EM[0.12] == 0); NDMG = CP & ~DMG
print('\n(9-1) AUROC [저항 vs flip | prop고정]  [C+손상 vs 비손상 | prop고정]  + 중앙값')
for nm, s in [('proj', PROJ.mean(1)), ('cos', COS.mean(1))]:
    m = FL | RES
    print(f'  {nm:4s}  저항 {roc_auc_score(RES[m].astype(int), s[m]):.3f} | {auc_within_prop(s, RES, FL)}'
          f'   C+손상 {roc_auc_score(DMG[CP].astype(int), s[CP]):.3f} | {auc_within_prop(s, DMG, NDMG)}')
    print('        중앙값 ' + ' / '.join(f'{k} {np.median(s[v]):.3f}' for k, v in
          [('flip', FL), ('저항', RES), ('C+손상', DMG), ('C+비손상', NDMG),
           ('K+C+', CELL == 'K+C+'), ('K−C+', CELL == 'K−C+'), ('K−C−', CELL == 'K−C−')]))

# (9-2) 정책: 목표 사영 t(train 분위수)까지 가는 데 필요한 ρ_i = mean_L[(t_L − proj)/MED], [0, ρmax]로 자르고 격자에 맞춤
RG = [0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12]
EMA, ar = np.stack([EM[r] for r in RG]), np.arange(len(F))
snap = lambda rho: EMA[np.clip(np.rint(rho / 0.02), 0, 6).astype(int), ar]
def need_rho(tq, tr):
    t = np.quantile(PROJ[tr], tq, axis=0)                          # 층별 목표 (train 행만)
    return ((t[None, :] - PROJ) / MEDR).mean(1)
def nested_geo():
    out, rho_used, pk = np.zeros(len(F)), np.zeros(len(F)), []
    for f in range(5):
        tr, te = FOLD != f, FOLD == f
        best = (EA[tr].mean(), None, None)                          # 기본값: A0 (ρ0.06)
        for tq in (0.5, 0.6, 0.7, 0.8, 0.9, 0.95):
            r0 = need_rho(tq, tr)
            for rmax in (0.08, 0.10, 0.12):
                e = snap(np.clip(r0, 0, rmax))
                if (e - E0)[tr & CP].mean() < -0.005: continue
                if e[tr].mean() > best[0]: best = (e[tr].mean(), tq, rmax)
        _, tq, rmax = best
        if tq is None: e, r = EA, np.full(len(F), 0.06)
        else: r = np.clip(need_rho(tq, tr), 0, rmax); e = snap(r)
        out[te], rho_used[te] = e[te], r[te]; pk.append('A0' if tq is None else f't{tq}/≤{rmax}')
    return out, rho_used, pk

# 기준: 같은 규칙의 nested 상수 ρ (Cell 7과 동일 정의)
def nested_const():
    out = np.zeros(len(F))
    for f in range(5):
        tr, te = FOLD != f, FOLD == f
        ok = [(EMA[k][tr].mean(), k) for k in range(1, 7) if (EMA[k] - E0)[tr & CP].mean() >= -0.005]
        k = max(ok)[1] if ok else 3
        out[te] = EMA[k][te]
    return out
base_c = nested_const()
geo, rho_g, pk_g = nested_geo()
print(f'\n(9-2) 상수 ρ nested EM {base_c.mean():.4f}  (Cell 7: 0.5915)')
print(f'      같은 도착점 EM {geo.mean():.4f} | − 상수 {boot(geo - base_c, ALL)} | C+ vs base {boot(geo - E0, CP)}')
print(f'      선택 {pk_g}')
print('      셀별 평균 ρ:', {c: round(float(rho_g[CELL == c].mean()), 3) for c in ['K+C+', 'K+C−', 'K−C+', 'K−C−']})
print('      셀별 Δ vs 상수:', {c: round(float((geo - base_c)[CELL == c].mean()), 4) for c in ['K+C+', 'K+C−', 'K−C+', 'K−C−']})
pd.DataFrame({'row': ar, 'em_geo': geo, 'rho_geo': rho_g}).to_parquet(OUT/'e18_geo_policy.parquet', index=False)

(9-0) cap_bs1 키: {'HL13': (1224, 32, 128), 'H13': (1224, 4096), 'HW': (1224, 5, 4096)}

(9-1) AUROC [저항 vs flip | prop고정]  [C+손상 vs 비손상 | prop고정]  + 중앙값
  proj  저항 0.376 | (0.35, 7)   C+손상 0.537 | (0.57, 9)
        중앙값 flip -2.265 / 저항 -2.761 / C+손상 -2.678 / C+비손상 -2.924 / K+C+ -2.743 / K−C+ -3.291 / K−C− -2.767
  cos   저항 0.381 | (0.347, 7)   C+손상 0.529 | (0.569, 9)
        중앙값 flip -0.227 / 저항 -0.273 / C+손상 -0.268 / C+비손상 -0.285 / K+C+ -0.266 / K−C+ -0.323 / K−C− -0.277

(9-2) 상수 ρ nested EM 0.5915  (Cell 7: 0.5915)
      같은 도착점 EM 0.5997 | − 상수 (0.0082, [0.0024, 0.0147]) | C+ vs base (-0.0033, [-0.0212, 0.0147])
      선택 ['A0', 'A0', 't0.9/≤0.08', 'A0', 'A0']
      셀별 평균 ρ: {'K+C+': 0.063, 'K+C−': 0.062, 'K−C+': 0.064, 'K−C−': 0.063}
      셀별 Δ vs 상수: {'K+C+': 0.0032, 'K+C−': 0.019, 'K−C+': 0.0, 'K−C−': 0.0101}


In [ ]:
## Cell 9 (추가) — fold별 차이 분해
print({f: round(float((geo - base_c)[FOLD == f].mean()), 4) for f in range(5)}, '| 선택', pk_g)

{0: 0.0, 1: 0.0, 2: 0.0041, 3: 0.0, 4: 0.0369} | 선택 ['A0', 'A0', 't0.9/≤0.08', 'A0', 'A0']
